# Notebook 4: Spatial Features Extraction

## Purpose
Extracts static spatial features (elevation and land cover) for all 162 unique monitoring locations.

## Input
- `unique_locations_for_extraction.csv` (162 unique sites)

## Process
- **Elevation:** Point-sampled from Copernicus GLO-30 DEM (30m resolution) at each site's coordinates
- **Land Cover:** Created 250m and 500m circular buffers around each site, calculated pixel counts for each land cover class within buffers, assigned majority class to each buffer
  - 250m buffer: Captures immediate riparian zone
  - 500m buffer: Captures broader catchment area

## Output
- `elevation_features.csv` (162 rows × 3 columns: Lat, Lon, Elevation)
- `land_cover_features.csv` (162 rows × 20 columns: Lat, Lon, 9 classes × 2 buffers)

**Note:** Buffer-based extraction (majority voting) had stronger correlation with water quality (r=0.54 for 250m buffer) compared to point-based extraction (r=0.11), which often classified river sampling points as "Water" (uninformative).

Cell 1: Install Required Packages

In [1]:
!pip install "numpy<2.0" pandas rasterio pystac-client planetary-computer tqdm

INFO: pip is looking at multiple versions of rasterio to determine which version is compatible with other requirements. This could take a while.
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl.metadata (6.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 1.9 MB/s eta 0:00:00m eta 0:00:010:00:01
Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: rasterio
    Found existing installation: rasterio 1.5.0
    Uninstalling rasterio-1.5.0:
      Successfully uninstalled rasterio-1.5.0


Cell 2: Import Libraries

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import rasterio
import pystac_client
import planetary_computer as pc
from tqdm import tqdm
import base64
from io import BytesIO

print("✓ Libraries imported successfully!")
print(f"pandas version: {pd.__version__}")
print(f"rasterio version: {rasterio.__version__}")

✓ Libraries imported successfully!
pandas version: 3.0.2
rasterio version: 1.4.4


Cell 3: Load Input Data

In [5]:
# Load unique locations for extraction
print("Loading unique locations for extraction...")
locations_df = pd.read_csv("unique_locations_for_extraction.csv")


print(f"DATA LOADED")
print(f"{'='*60}")
print(f"Total unique locations: {len(locations_df)}")
print(f"Columns: {list(locations_df.columns)}")
print(f"\nData preview:")
print(locations_df.head())
print(f"\n{'='*60}")
print(f"Coordinate ranges:")
print(f"Latitude:  {locations_df['Latitude'].min():.4f} to {locations_df['Latitude'].max():.4f}")
print(f"Longitude: {locations_df['Longitude'].min():.4f} to {locations_df['Longitude'].max():.4f}")
print(f"{'='*60}")

Loading unique locations for extraction...
DATA LOADED
Total unique locations: 162
Columns: ['Latitude', 'Longitude']

Data preview:
    Latitude  Longitude
0 -34.405833  19.600556
1 -34.329722  18.990278
2 -34.250556  20.992500
3 -34.092222  21.295278
4 -34.075556  20.145556

Coordinate ranges:
Latitude:  -34.4058 to -22.2256
Longitude: 17.7303 to 32.3250


Cell 4: Define Elevation Extraction Function

In [7]:
def extract_elevation_for_points(df: pd.DataFrame) -> pd.Series:
    """
    Extract elevation (meters above sea level) at each (Latitude, Longitude) 
    using Copernicus DEM GLO-30 from Microsoft Planetary Computer.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with 'Latitude' and 'Longitude' columns
    
    Returns:
    --------
    pd.Series
        Series with elevation values (meters) aligned to input DataFrame index
    
    Notes:
    ------
    - Uses tile caching to avoid repeated downloads for nearby locations
    - Handles nodata value (-32768) from Copernicus DEM
    - Returns NaN for locations outside tile coverage or with extraction errors
    """
    # Connect to Microsoft Planetary Computer STAC API
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    
    # Define bounding box for South Africa region
    # [min_lon, min_lat, max_lon, max_lat]
    bbox = [14.97, -35.18, 32.79, -21.72]
    
    # Search for Copernicus DEM tiles covering the study region
    print("Searching for Copernicus DEM tiles...")
    search = catalog.search(
        collections=["cop-dem-glo-30"],
        bbox=bbox,
    )
    items = list(search.get_items())
    print(f"✓ Found {len(items)} Copernicus DEM tiles covering South Africa")
    
    # Create bounding boxes for each tile for efficient point-in-tile checks
    def get_bbox(item):
        """Extract bounding box from STAC item geometry"""
        coords = item.geometry["coordinates"][0]
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        return min(lons), min(lats), max(lons), max(lats)
    
    item_bboxes = [(item, get_bbox(item)) for item in items]
    
    def find_tile_for_point(lon, lat):
        """Find which DEM tile contains the given point"""
        for item, (min_lon, min_lat, max_lon, max_lat) in item_bboxes:
            if min_lon <= lon <= max_lon and min_lat <= lat <= max_lat:
                return item
        return None
    
    # Asset key for elevation band in Copernicus DEM
    elevation_asset_key = "data"
    
    # Tile cache: tile_id -> opened rasterio dataset
    # This avoids re-downloading the same tile for nearby points
    tile_cache = {}
    
    # Extract elevation for each point
    elevations = []
    failed_points = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting elevation"):
        lat = row["Latitude"]
        lon = row["Longitude"]
        
        # Find which tile contains this point
        item = find_tile_for_point(lon, lat)
        if item is None:
            elevations.append(np.nan)
            failed_points.append((idx, lat, lon, "No tile found"))
            continue
        
        # Get elevation asset from the tile
        asset = item.assets.get(elevation_asset_key) or item.assets.get("elevation")
        if asset is None:
            elevations.append(np.nan)
            failed_points.append((idx, lat, lon, "No elevation asset"))
            continue
        
        tile_id = item.id
        try:
            # Load tile from cache or download if not cached
            if tile_id not in tile_cache:
                signed_asset = pc.sign(asset)
                tile_cache[tile_id] = rasterio.open(signed_asset.href)
            
            src = tile_cache[tile_id]
            
            # Sample elevation at (lon, lat) coordinates
            # rasterio.sample expects (x, y) = (lon, lat) in EPSG:4326
            vals = list(src.sample([(lon, lat)]))
            
            if vals:
                v = float(vals[0][0])
                # Copernicus DEM uses -32768 as nodata value
                if v <= -32768:
                    elevations.append(np.nan)
                    failed_points.append((idx, lat, lon, "Nodata value"))
                else:
                    elevations.append(v)
            else:
                elevations.append(np.nan)
                failed_points.append((idx, lat, lon, "No sample value"))
                
        except Exception as e:
            elevations.append(np.nan)
            failed_points.append((idx, lat, lon, f"Error: {str(e)}"))
    
    # Close all cached rasterio datasets
    for ds in tile_cache.values():
        ds.close()
    
    # Report extraction statistics
    print(f"\n{'='*60}")
    print(f"EXTRACTION COMPLETE")
    print(f"{'='*60}")
    print(f"Total points processed: {len(elevations)}")
    print(f"Successful extractions: {sum(~pd.isna(elevations))}")
    print(f"Failed extractions: {sum(pd.isna(elevations))}")
    print(f"Tiles cached: {len(tile_cache)}")
    
    if failed_points:
        print(f"\n⚠ Warning: {len(failed_points)} points failed extraction")
        print("First 5 failed points:")
        for fp in failed_points[:5]:
            print(f"  Index {fp[0]}: ({fp[1]:.4f}, {fp[2]:.4f}) - {fp[3]}")
    
    return pd.Series(elevations, index=df.index)

Cell 5: Extract Elevation Data

In [9]:
print("Starting elevation extraction...")

# Extract elevation for all unique locations
elevation_series = extract_elevation_for_points(locations_df)

# Add elevation column to dataframe
locations_df['elevation'] = elevation_series

print("\n✓ Elevation extraction complete!")

Starting elevation extraction...
Searching for Copernicus DEM tiles...
✓ Found 226 Copernicus DEM tiles covering South Africa


Extracting elevation: 100%|███████████████████| 162/162 [21:23<00:00,  7.92s/it]



EXTRACTION COMPLETE
Total points processed: 162
Successful extractions: 162
Failed extractions: 0
Tiles cached: 73

✓ Elevation extraction complete!


Cell 6: Data Quality Checks

In [11]:
# Quality checks
print("ELEVATION QUALITY CHECKS")

valid = locations_df['elevation'].dropna()
missing = locations_df['elevation'].isna().sum()

print(f"Missing: {missing}/{len(locations_df)} ({missing/len(locations_df)*100:.1f}%)")
print(f"Range: {valid.min():.1f}m to {valid.max():.1f}m (expected: 0-3,500m)")
print(f"Duplicates: {locations_df.duplicated(subset=['Latitude', 'Longitude']).sum()}")

print(f"\nSummary:\n{locations_df['elevation'].describe()}")

ELEVATION QUALITY CHECKS
Missing: 0/162 (0.0%)
Range: 2.5m to 1590.2m (expected: 0-3,500m)
Duplicates: 0

Summary:
count     162.000000
mean      747.624534
std       527.890521
min         2.463319
25%       181.283657
50%       828.148010
75%      1247.926025
max      1590.156738
Name: elevation, dtype: float64


Cell 7: Visualize Elevation Distribution

In [13]:
# Elevation distribution
print("ELEVATION DISTRIBUTION")

elev = locations_df['elevation'].dropna()

# Quartiles
print(f"\nQuartiles: Min={elev.min():.0f}m | Q1={elev.quantile(0.25):.0f}m | "
      f"Median={elev.median():.0f}m | Q3={elev.quantile(0.75):.0f}m | Max={elev.max():.0f}m")
print(f"Mean={elev.mean():.0f}m | Std={elev.std():.0f}m")

# Range distribution
print("\nRange distribution:")
ranges = [(0, 500, "Coastal"), (500, 1000, "Low-Mid"), (1000, 1500, "Mid"), 
          (1500, 2000, "High"), (2000, 4000, "Very High")]

for min_e, max_e, label in ranges:
    count = ((elev >= min_e) & (elev < max_e)).sum()
    print(f"  {min_e:4d}-{max_e:4d}m ({label:10s}): {count:3d} ({count/len(elev)*100:4.1f}%)")

ELEVATION DISTRIBUTION

Quartiles: Min=2m | Q1=181m | Median=828m | Q3=1248m | Max=1590m
Mean=748m | Std=528m

Range distribution:
     0- 500m (Coastal   ):  65 (40.1%)
   500-1000m (Low-Mid   ):  33 (20.4%)
  1000-1500m (Mid       ):  55 (34.0%)
  1500-2000m (High      ):   9 ( 5.6%)
  2000-4000m (Very High ):   0 ( 0.0%)


Cell 8: Save Output File

In [15]:
# Save elevation features to CSV
output_filename = "elevation_features.csv"
locations_df.to_csv(output_filename, index=False)

print(f"{'-'*60}")
print(f"Filename: {output_filename}")
print(f"Rows: {len(locations_df)}")
print(f"Columns: {list(locations_df.columns)}")
print(f"{'-'*60}")
# create downloadable link for Snowflake environment
def create_download_link(filename):
    """Create base64 encoded download link for file"""
    with open(filename, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return f'<a href="data:file/csv;base64,{b64}" download="{filename}">Download {filename}</a>'

from IPython.display import HTML, display


display(HTML(create_download_link(output_filename)))


------------------------------------------------------------
Filename: elevation_features.csv
Rows: 162
Columns: ['Latitude', 'Longitude', 'elevation']
------------------------------------------------------------


ESA WORLDCOVER LAND COVER EXTRACTION

Cell 1: Extract Land Cover for Different Buffer Sizes

In [17]:
def extract_all_buffer_sizes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extract land cover for multiple buffer sizes (100m, 250m, 500m, 1000m).
    Returns DataFrame with dominant class and water % for each buffer.
    """
    import rasterio
    from rasterio.windows import from_bounds
    import math
    
    # Connect to Microsoft Planetary Computer
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    
    # South Africa bounding box
    bbox = [14.97, -35.18, 32.79, -21.72]
    search = catalog.search(collections=["esa-worldcover"], bbox=bbox)
    items = list(search.get_items())
    print(f"✓ Found {len(items)} ESA WorldCover tiles")
    
    def get_bbox(item):
        coords = item.geometry["coordinates"][0]
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        return min(lons), min(lats), max(lons), max(lats)
    
    item_bboxes = [(item, get_bbox(item)) for item in items]
    
    def find_tile_for_point(lon, lat):
        for item, (min_lon, min_lat, max_lon, max_lat) in item_bboxes:
            if min_lon <= lon <= max_lon and min_lat <= lat <= max_lat:
                return item
        return None
    
    # Land cover class names
    lc_names = {
        10: "Tree cover", 20: "Shrubland", 30: "Grassland", 
        40: "Cropland", 50: "Built-up", 60: "Bare/sparse",
        70: "Snow/ice", 80: "Water", 90: "Wetland", 
        95: "Mangroves", 100: "Moss/lichen"
    }
    
    # Buffer sizes to extract
    buffer_sizes = [100, 250, 500, 1000]
    meters_per_degree_lat = 111000
    tile_cache = {}
    results = []
    
    print(f"\nExtracting land cover for buffer sizes: {buffer_sizes} meters")
    print(f"This creates circular areas around each point to capture surrounding land use.")
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Multi-buffer extraction"):
        lat = row["Latitude"]
        lon = row["Longitude"]
        
        # Find tile containing this point
        item = find_tile_for_point(lon, lat)
        if item is None:
            result = {'Latitude': lat, 'Longitude': lon}
            for size in buffer_sizes:
                result[f'LC_{size}m'] = np.nan
                result[f'LC_{size}m_name'] = 'No tile'
                result[f'Water%_{size}m'] = np.nan
            results.append(result)
            continue
        
        tile_id = item.id
        asset = item.assets.get("map")
        
        if asset is None:
            result = {'Latitude': lat, 'Longitude': lon}
            for size in buffer_sizes:
                result[f'LC_{size}m'] = np.nan
                result[f'LC_{size}m_name'] = 'No asset'
                result[f'Water%_{size}m'] = np.nan
            results.append(result)
            continue
        
        try:
            # Load tile from cache
            if tile_id not in tile_cache:
                signed_asset = pc.sign(asset)
                tile_cache[tile_id] = rasterio.open(signed_asset.href)
            
            src = tile_cache[tile_id]
            
            # Extract for each buffer size
            result = {'Latitude': lat, 'Longitude': lon}
            
            for buffer_meters in buffer_sizes:
                # Convert buffer from meters to degrees
                buffer_deg_lat = buffer_meters / meters_per_degree_lat
                meters_per_degree_lon = meters_per_degree_lat * math.cos(math.radians(lat))
                buffer_deg_lon = buffer_meters / meters_per_degree_lon
                
                # Create bounding box for this buffer zone
                bbox_buffer = [
                    lon - buffer_deg_lon,
                    lat - buffer_deg_lat,
                    lon + buffer_deg_lon,
                    lat + buffer_deg_lat
                ]
                
                # Read all pixels within the buffer window
                window = from_bounds(*bbox_buffer, transform=src.transform)
                data = src.read(1, window=window)
                
                # Count pixels by land cover class
                values, counts = np.unique(data[data > 0], return_counts=True)
                
                if len(values) == 0:
                    result[f'LC_{buffer_meters}m'] = np.nan
                    result[f'LC_{buffer_meters}m_name'] = 'No data'
                    result[f'Water%_{buffer_meters}m'] = np.nan
                else:
                    # Find dominant class (most common)
                    dominant_idx = np.argmax(counts)
                    dominant_class = int(values[dominant_idx])
                    
                    # Calculate water percentage
                    water_idx = np.where(values == 80)[0]
                    water_pct = (counts[water_idx[0]] / counts.sum() * 100) if len(water_idx) > 0 else 0.0
                    
                    result[f'LC_{buffer_meters}m'] = dominant_class
                    result[f'LC_{buffer_meters}m_name'] = lc_names.get(dominant_class, 'Unknown')
                    result[f'Water%_{buffer_meters}m'] = round(water_pct, 1)
            
            results.append(result)
            
        except Exception as e:
            result = {'Latitude': lat, 'Longitude': lon}
            for size in buffer_sizes:
                result[f'LC_{size}m'] = np.nan
                result[f'LC_{size}m_name'] = 'Error'
                result[f'Water%_{size}m'] = np.nan
            results.append(result)
    
    # Close all cached tiles
    for ds in tile_cache.values():
        ds.close()
    
    results_df = pd.DataFrame(results)
    
    # Print summary statistics
    print(f"\n{'='*80}")
    print("MULTI-BUFFER EXTRACTION COMPLETE")
    print(f"{'='*80}")
    
    for size in buffer_sizes:
        water_count = (results_df[f'LC_{size}m'] == 80).sum()
        mean_water = results_df[f'Water%_{size}m'].mean()
        valid_count = results_df[f'LC_{size}m'].notna().sum()
        print(f"{size:4d}m buffer: {valid_count:3d} extracted | "
              f"{water_count:2d} water-dominated | Mean water: {mean_water:5.1f}%")
    
    print(f"{'='*80}\n")
    
    return results_df

# Run extraction
print("Starting multi-buffer land cover extraction...")
print("This extracts land cover at 100m, 250m, 500m, and 1000m buffers.")

multi_buffer_results = extract_all_buffer_sizes(locations_df)

# Preview results
print("Preview of multi-buffer results:")
preview_cols = ['Latitude', 'Longitude', 
                'LC_100m_name', 'Water%_100m',
                'LC_250m_name', 'Water%_250m', 
                'LC_500m_name', 'Water%_500m',
                'LC_1000m_name', 'Water%_1000m']
print(multi_buffer_results[preview_cols].head(10))

print("\n✓ Multi-buffer extraction complete!")

Starting multi-buffer land cover extraction...
This extracts land cover at 100m, 250m, 500m, and 1000m buffers.
✓ Found 62 ESA WorldCover tiles

Extracting land cover for buffer sizes: [100, 250, 500, 1000] meters
This creates circular areas around each point to capture surrounding land use.


Multi-buffer extraction: 100%|████████████████| 162/162 [01:27<00:00,  1.85it/s]


MULTI-BUFFER EXTRACTION COMPLETE
 100m buffer: 162 extracted | 21 water-dominated | Mean water:  14.1%
 250m buffer: 162 extracted |  9 water-dominated | Mean water:   7.2%
 500m buffer: 162 extracted |  1 water-dominated | Mean water:   3.9%
1000m buffer: 162 extracted |  0 water-dominated | Mean water:   2.3%

Preview of multi-buffer results:
    Latitude  Longitude LC_100m_name  Water%_100m LC_250m_name  Water%_250m  \
0 -34.405833  19.600556    Grassland          0.0    Grassland          0.0   
1 -34.329722  18.990278   Tree cover          0.0    Shrubland          1.2   
2 -34.250556  20.992500    Shrubland          0.0    Shrubland          0.0   
3 -34.092222  21.295278    Grassland          0.0    Grassland          0.0   
4 -34.075556  20.145556    Shrubland          0.0    Shrubland          0.2   
5 -34.065833  20.404167   Tree cover         19.1    Grassland         12.0   
6 -34.039722  22.133333   Tree cover          0.0   Tree cover          0.0   
7 -34.031944  22.053

Cell 2: Correlation Analysis and Buffer Size Selection

In [22]:
# Landcover class stability analysis
print("-"*80)
print("1. CLASS STABILITY ACROSS BUFFER SIZES")
print("-"*80)

same_all = ((multi_buffer_results['LC_100m'] == multi_buffer_results['LC_250m']) & 
            (multi_buffer_results['LC_250m'] == multi_buffer_results['LC_500m']) & 
            (multi_buffer_results['LC_500m'] == multi_buffer_results['LC_1000m']))
print(f"Locations with same class at ALL buffers: {same_all.sum()} / {len(multi_buffer_results)} ({same_all.sum()/len(multi_buffer_results)*100:.1f}%)")

same_100_250 = (multi_buffer_results['LC_100m'] == multi_buffer_results['LC_250m']).sum()
same_250_500 = (multi_buffer_results['LC_250m'] == multi_buffer_results['LC_500m']).sum()
same_500_1000 = (multi_buffer_results['LC_500m'] == multi_buffer_results['LC_1000m']).sum()

print(f"\nStability between adjacent buffers:")
print(f"  100m → 250m:  {same_100_250}/{len(multi_buffer_results)} stable ({same_100_250/len(multi_buffer_results)*100:.1f}%)")
print(f"  250m → 500m:  {same_250_500}/{len(multi_buffer_results)} stable ({same_250_500/len(multi_buffer_results)*100:.1f}%)")
print(f"  500m → 1000m: {same_500_1000}/{len(multi_buffer_results)} stable ({same_500_1000/len(multi_buffer_results)*100:.1f}%)")
print("\n→ Higher stability = less differentiation between buffer sizes")

# 2. CORRELATION ANALYSIS (KEY METRIC)
print("\n" + "-"*80)
print("2. CORRELATION BETWEEN BUFFER SIZES (Class Codes)")
print("-"*80)

corr_matrix = multi_buffer_results[['LC_100m', 'LC_250m', 'LC_500m', 'LC_1000m']].corr()
print("\nCorrelation matrix:")
print(corr_matrix.round(3))

print("\nKEY CORRELATIONS:")
print(f"  100m ↔ 250m:  r = {corr_matrix.loc['LC_100m', 'LC_250m']:.3f} (moderate similarity)")
print(f"  250m ↔ 500m:  r = {corr_matrix.loc['LC_250m', 'LC_500m']:.3f} (moderate - GOOD for 2 features)")
print(f"  500m ↔ 1000m: r = {corr_matrix.loc['LC_500m', 'LC_1000m']:.3f} (high - TOO SIMILAR)")
print(f"  100m ↔ 500m:  r = {corr_matrix.loc['LC_100m', 'LC_500m']:.3f} (low - maximally different)")

print("\n→ Lower correlation = more independent information")
print("→ Too low (<0.3) = might be measuring different things")
print("→ Too high (>0.7) = redundant information")

# 3. WATER CONTAMINATION
print("\n" + "-"*80)
print("3. WATER PERCENTAGE (Signal vs Noise)")
print("-"*80)

for col in ['Water%_100m', 'Water%_250m', 'Water%_500m', 'Water%_1000m']:
    mean_water = multi_buffer_results[col].mean()
    buffer_size = col.split('_')[1]
    print(f"  {buffer_size:8s}: Mean = {mean_water:5.1f}%")

print("\n→ Lower water % = better signal (less contamination from river itself)")
print("→ Our sampling points are IN rivers, so we want buffers that capture surrounding land")

# 4. HIGH-RISK DETECTION
print("\n" + "-"*80)
print("4. HIGH-RISK LOCATION DETECTION (Cropland + Built-up)")
print("-"*80)

high_risk_codes = [40, 50]
for col_code, buffer_name in [('LC_100m', '100m'), ('LC_250m', '250m'), 
                                ('LC_500m', '500m'), ('LC_1000m', '1000m')]:
    high_risk_count = multi_buffer_results[col_code].isin(high_risk_codes).sum()
    print(f"  {buffer_name:8s}: {high_risk_count:3d} locations ({high_risk_count/len(multi_buffer_results)*100:5.1f}%)")

print("\n→ Larger buffers detect more high-risk sites (captures wider catchment)")
print("→ But diminishing returns: 500m and 1000m find the SAME number (22)")



--------------------------------------------------------------------------------
1. CLASS STABILITY ACROSS BUFFER SIZES
--------------------------------------------------------------------------------
Locations with same class at ALL buffers: 76 / 162 (46.9%)

Stability between adjacent buffers:
  100m → 250m:  121/162 stable (74.7%)
  250m → 500m:  125/162 stable (77.2%)
  500m → 1000m: 131/162 stable (80.9%)

→ Higher stability = less differentiation between buffer sizes

--------------------------------------------------------------------------------
2. CORRELATION BETWEEN BUFFER SIZES (Class Codes)
--------------------------------------------------------------------------------

Correlation matrix:
          LC_100m  LC_250m  LC_500m  LC_1000m
LC_100m     1.000    0.533    0.190     0.134
LC_250m     0.533    1.000    0.542     0.447
LC_500m     0.190    0.542    1.000     0.787
LC_1000m    0.134    0.447    0.787     1.000

KEY CORRELATIONS:
  100m ↔ 250m:  r = 0.533 (moderate sim

Cell 3: Extract Final Landcover Features for ML

In [24]:
# Create dataset with land cover features
land_cover_features = pd.DataFrame({
    'Latitude': locations_df['Latitude'],
    'Longitude': locations_df['Longitude'],
    'land_cover_250m': multi_buffer_results['LC_250m_name'],
    'land_cover_500m': multi_buffer_results['LC_500m_name']
})

print(f"\nDataset shape: {land_cover_features.shape}")
print(f"Columns: {list(land_cover_features.columns)}")

print("\n250m Buffer Distribution:")
lc_250 = land_cover_features['land_cover_250m'].value_counts()
for class_name, count in lc_250.items():
    pct = count / len(land_cover_features) * 100
    print(f"  {class_name:20s}: {count:3d} ({pct:5.1f}%)")

print("\n500m Buffer Distribution:")
lc_500 = land_cover_features['land_cover_500m'].value_counts()
for class_name, count in lc_500.items():
    pct = count / len(land_cover_features) * 100
    print(f"  {class_name:20s}: {count:3d} ({pct:5.1f}%)")

# Quality check
complete = land_cover_features.notna().all(axis=1).sum()
print(f"\n✓ Complete rows: {complete}/{len(land_cover_features)} ({complete/len(land_cover_features)*100:.1f}%)")

# Save
output_filename = "land_cover_features.csv"
land_cover_features.to_csv(output_filename, index=False)

print(f"\n✓ Saved: {output_filename}")

# Download link
from IPython.display import HTML, display

def create_download_link(filename):
    with open(filename, 'rb') as f:
        data = f.read()
    b64 = base64.b64encode(data).decode()
    return f'<a href="data:file/csv;base64,{b64}" download="{filename}">Download {filename}</a>'

display(HTML(create_download_link(output_filename)))
print("="*80)


Dataset shape: (162, 4)
Columns: ['Latitude', 'Longitude', 'land_cover_250m', 'land_cover_500m']

250m Buffer Distribution:
  Grassland           :  58 ( 35.8%)
  Tree cover          :  44 ( 27.2%)
  Shrubland           :  33 ( 20.4%)
  Cropland            :  13 (  8.0%)
  Water               :   9 (  5.6%)
  Built-up            :   3 (  1.9%)
  Bare/sparse         :   2 (  1.2%)

500m Buffer Distribution:
  Grassland           :  63 ( 38.9%)
  Shrubland           :  41 ( 25.3%)
  Tree cover          :  30 ( 18.5%)
  Cropland            :  17 ( 10.5%)
  Built-up            :   5 (  3.1%)
  Bare/sparse         :   5 (  3.1%)
  Water               :   1 (  0.6%)

✓ Complete rows: 162/162 (100.0%)

✓ Saved: land_cover_features.csv
